# RSNA EXP-002 A3 fold0 GPU probe

唯一变量: baseline pseudo-label → A3 Contusion-only conservative rule

- 其余 11 targets 与 EXP-001 完全一致
- 无 A0 dual-pol / 无 Synovitis inheritance / 无 Effusion 改动
- label sha256: `8d95e710d7e19ba58a5162071703b2388b2a84edc453e5686f36a4c0af7de17a`
- 超参严格复用 EXP-001 fold0: sagittal ResNet34 n_clips=2 224 batch=8 ep=3 AdamW 3e-4 wd=1e-4 Cosine GroupKFold(5) --only-fold 0
- 不自动提交; 与 EXP-001 fold0 LB=0.663 对照


In [ ]:
# 提交模式:禁网(enable_internet:false),torch 2.2.0+cu118 + nvidia-cu11 依赖全部离线安装
# 坑1:Kaggle 挂载把文件名中 '+' 去掉(2.2.0cu118=非法 PEP440)→ 复制为合法名
# 坑2:torch 2.2.0+cu118 的 CUDA 库在 nvidia-*-cu11 依赖包中(不在 torch wheel 内),
#     只装 torch 会缺 libcudart(v8/v9 实证)→ 必须显式安装全部 13 个 wheel
# 坑3:/kaggle/working 会被收集为 kernel 输出 → 装完立即删除
import subprocess, sys, os, glob, shutil
wh = sorted(glob.glob('/kaggle/input/**/*.whl', recursive=True))
print('挂载 wheel 共', len(wh), '个')
for w in wh: print('  ', os.path.basename(w))
assert len(wh) >= 13, 'FAIL: 离线 wheel 不足(应 13: torch+torchvision+11 nvidia-cu11)'
fixed = []
for w in wh:
    base = os.path.basename(w)
    dst = os.path.join('/kaggle/working', base.replace('cu118', '+cu118', 1) if 'cu118' in base else base)
    shutil.copy(w, dst)
    fixed.append(dst)
print('安装 wheel:', [os.path.basename(f) for f in fixed])
subprocess.check_call([sys.executable,"-m","pip","uninstall","-y","-q","torch","torchvision","torchaudio"])
# 显式安装全部 wheel(--no-deps --no-index: 无版本解析/无联网)
subprocess.check_call([sys.executable,"-m","pip","install","-q","--no-deps","--no-index"] + fixed)
for f in fixed: os.remove(f)
import numpy, torch, torchvision, sklearn, pandas
print("torch", torch.__version__, "cuda", torch.version.cuda, "| numpy", numpy.__version__)
assert torch.cuda.is_available()
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# preflight:test DICOM 布局抽样(防训完才暴露 test 读不到,浪费额度)
# 扫描兼容 1/2/3 层挂载:/kaggle/input/<slug>/、/<slug>/<slug>、/competitions/<slug>;
# 跳过 train_series/test_series 大目录(内部不可能有 test.csv)
import os
base='/kaggle/input'
def _find(b, depth=0, max_depth=3):
    if not os.path.isdir(b): return None
    try: entries = sorted(os.listdir(b))
    except OSError: return None
    for e in entries:
        p = os.path.join(b, e)
        if not os.path.isdir(p): continue
        if e in ('train_series', 'test_series'): continue
        if os.path.isfile(os.path.join(p, 'test.csv')) and os.path.isdir(os.path.join(p, 'test_series')):
            return p
        if depth < max_depth - 1:
            r = _find(p, depth + 1)
            if r: return r
    return None
root = _find(base)
if root is None:
    raise SystemExit('FAIL: 未找到含 test_series+test.csv 的比赛挂载(已扫 /kaggle/input 3 层)')
print('root =', root)
ts=os.path.join(root,'test_series'); n_dcm=0
for st in sorted(os.listdir(ts))[:3]:
    sp=os.path.join(ts,st)
    if not os.path.isdir(sp): continue
    for se in sorted(os.listdir(sp))[:3]:
        sdir=os.path.join(sp,se)
        if not os.path.isdir(sdir): continue
        k=len([f for f in os.listdir(sdir) if f.lower().endswith('.dcm')])
        print(' study', st[:14], 'series', se[:14], 'dcm:', k); n_dcm+=k
print('preflight:', 'OK' if n_dcm>0 else 'FAIL: test DICOM 为空')
assert n_dcm>0


In [ ]:
# write A3 label manifest into artifact root (before training)
import json, os, hashlib
os.makedirs("/kaggle/working/artifacts/exp_002_a3_fold0", exist_ok=True)
manifest = {
  "experiment_id": "exp_002_a3_fold0",
  "label_variant": "A3",
  "description": "baseline + Contusion-only conservative (edema/degenerative/subchondral without trauma -> silent)",
  "sha256": "8d95e710d7e19ba58a5162071703b2388b2a84edc453e5686f36a4c0af7de17a",
  "parent": "rsna_report_labels.py baseline",
  "changes": ["Contusion scorer only"],
  "not_included": ["A0 dual-pol", "Effusion negation change", "Synovitis inheritance"],
  "exp001_fold0_LB_reference": 0.663,
}
path = "/kaggle/working/artifacts/exp_002_a3_fold0/label_manifest.json"
with open(path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)
print("wrote", path)
# verify on-disk hash matches
p = "/kaggle/working/rsna_report_labels.py"


In [ ]:
import base64, os
parts = []
parts.append("...")
path = "/kaggle/working/rsna_v3.py"
with open(path, "w", encoding="utf-8") as f:
    f.write(base64.b64decode(parts[0]).decode("utf-8"))
print(f"wrote rsna_v3.py ({os.path.getsize(path)} bytes)")


In [ ]:
import base64, os, hashlib
parts = []
parts.append("...")
path = "/kaggle/working/rsna_report_labels.py"
with open(path, "w", encoding="utf-8") as f:
    f.write(base64.b64decode(parts[0]).decode("utf-8"))
expect = "8d95e710d7e19ba58a5162071703b2388b2a84edc453e5686f36a4c0af7de17a"
h = hashlib.sha256(open(path, "rb").read()).hexdigest()
print(f"wrote rsna_report_labels.py ({os.path.getsize(path)} bytes)")
print("sha256", h)
assert h == expect, f"LABEL HASH MISMATCH {{h}} != {{expect}}"
print("A3 label hash OK")


In [ ]:
import subprocess, sys, os, json
os.chdir("/kaggle/working")
cmd = [sys.executable, "-u", "rsna_v3.py",
       "--planes", "sagittal", "--epochs", "3", "--folds", "5",
       "--only-fold", "0", "--batch", "8", "--n-clips", "2", "--verbose",
       "--artifact-dir", "artifacts",
       "--experiment-id", "exp_002_a3_fold0"]
print("=== EXP-002 A3 fold0 only ===", flush=True)
print(" ".join(cmd), flush=True)
r = subprocess.run(cmd)
print("returncode", r.returncode, flush=True)

import pandas as pd
artifact_root = "/kaggle/working/artifacts/exp_002_a3_fold0"
for name in ["label_manifest.json", "config.json", "fold_assignment.csv", "test_uid.csv",
             "fold0/test_pred.npy", "fold0/test_uid.csv", "fold0/meta.json"]:
    p = os.path.join(artifact_root, name)
    print(f"[artifact] {name}: {'OK' if os.path.isfile(p) else 'MISSING'}", 
          os.path.getsize(p) if os.path.isfile(p) else "")

if os.path.isfile(f"{artifact_root}/fold0/meta.json"):
    meta = json.load(open(f"{artifact_root}/fold0/meta.json", encoding="utf-8"))
    print("[timing]", meta)

sub_path = "/kaggle/working/submission.csv"
if os.path.isfile(sub_path):
    sub = pd.read_csv(sub_path)
    print("[pred] submission shape", sub.shape)
    print("[pred] NaN", int(sub.isna().sum().sum()))
    import numpy as np
    pred = np.load(f"{artifact_root}/fold0/test_pred.npy")
    print("[pred] fold0 pred shape", pred.shape, "min", float(pred.min()), "max", float(pred.max()),
          "mean", float(pred.mean()), "std", float(pred.std()))
else:
    print("[pred] submission.csv missing")
